# Phase 1: Data Scoping, Segmentation & Diagnostics
### Adaptive SKU-Level Demand Forecasting & Inventory Decision Intelligence

**Dataset:** M5 Forecasting — Accuracy (Walmart retail sales, Kaggle)
**Scope:** California, FOODS category
**Goal:** Reduce the full item catalog to a defensible, analyzable subset and characterize its demand behavior before any forecasting or structural-break work begins.


## 1. Load Data and Verify Date Range

Before any analysis, the dataset's actual date range is checked programmatically rather than assumed — this matters because it rules out framing any later finding around events outside the data's real window (e.g. COVID).

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

sales = pd.read_csv("../data/raw/sales_train_evaluation.csv")
calendar = pd.read_csv("../data/raw/calendar.csv")
prices = pd.read_csv("../data/raw/sell_prices.csv")

print("Calendar date range:", calendar['date'].min(), "to", calendar['date'].max())
print("Sales shape:", sales.shape)
print("States:", sales['state_id'].unique())
print("Categories:", sales['cat_id'].unique())

Calendar date range: 2011-01-29 to 2016-06-19
Sales shape: (30490, 1947)
States: ['CA' 'TX' 'WI']
Categories: ['HOBBIES' 'HOUSEHOLD' 'FOODS']


**Confirmed:** the dataset spans 2011-01-29 to 2016-06-19 — well before COVID, so no external-shock narrative from that period is valid for this data.

## 2. Scope and Aggregate

Scoped to **California + FOODS** — the highest-volume, most consistent-demand category, and the largest state by store count. Aggregated across CA's stores to isolate demand-level signal from single-store noise (a store-specific stockout shouldn't look like a market-wide demand shift).

In [ ]:
scoped = sales[(sales['state_id'] == 'CA') & (sales['cat_id'] == 'FOODS')].copy()
print("SKU-store rows in scope:", scoped.shape[0])
print("Unique items:", scoped['item_id'].nunique())

# day columns are d_1, d_2, ... d_1941
day_cols = [c for c in scoped.columns if c.startswith('d_')]

# aggregate across CA's stores -> one row per item_id
item_level = scoped.groupby('item_id')[day_cols].sum().reset_index()
print("Aggregated item-level series:", item_level.shape)

SKU-store rows in scope: 5748
Unique items: 1437
Aggregated item-level series: (1437, 1942)


## 3. Reshape to Long Format

One row per item per date — the shape needed for time-series operations (ADF testing, decomposition, and later structural-break detection).

In [ ]:
long_df = item_level.melt(id_vars='item_id', var_name='d', value_name='sales')
long_df = long_df.merge(calendar[['d', 'date']], on='d', how='left')
long_df['date'] = pd.to_datetime(long_df['date'])
long_df = long_df.sort_values(['item_id', 'date']).reset_index(drop=True)
long_df.head()

       item_id    d  sales       date
0  FOODS_1_001  d_1      6 2011-01-29
1  FOODS_1_001  d_2      3 2011-01-30
2  FOODS_1_001  d_3      2 2011-01-31
3  FOODS_1_001  d_4      3 2011-02-01
4  FOODS_1_001  d_5      7 2011-02-02

## 4. Data Quality Fix: Pre-Launch Leading Zeros

Initial exploration of the raw series (seasonal decomposition on an early sample item) showed a flat, extended zero-sales stretch at the start of many series. This is **not genuine zero demand** — it's an artifact of the item not yet having launched. Left uncorrected, this would inflate variance estimates and risk false structural-break detection at launch dates in later phases.

`trim_leading_zeros()` cuts each item's series to its actual first-sale date.

In [ ]:
def trim_leading_zeros(df):
    """Keep only rows from an item's first non-zero sale date onward."""
    trimmed = []
    for item_id, grp in df.groupby('item_id'):
        grp = grp.sort_values('date')
        first_sale_idx = grp['sales'].to_numpy().nonzero()[0]
        if len(first_sale_idx) == 0:
            continue  # item never sold in this scope -> drop it
        start_pos = first_sale_idx[0]
        trimmed.append(grp.iloc[start_pos:])
    return pd.concat(trimmed, ignore_index=True)

n_before = long_df.shape[0]
long_df = trim_leading_zeros(long_df)
n_after = long_df.shape[0]
print(f"Rows before trim: {n_before}")
print(f"Rows after trim:  {n_after}")
print(f"Rows removed: {n_before - n_after} ({(1 - n_after/n_before)*100:.1f}%)")

Rows before trim: 2789217
Rows after trim:  2279871
Rows removed: 509346 (18.3%)


### Verification

Checked against the raw (pre-trim) data directly: how many items actually had a real leading-zero stretch (more than 30 days before first sale), and confirmed the fix on a real affected item rather than one that happened to need no correction.

In [ ]:
raw_check = sales[(sales['state_id']=='CA') & (sales['cat_id']=='FOODS')]
day_cols_raw = [c for c in raw_check.columns if c.startswith('d_')]
item_level_raw = raw_check.groupby('item_id')[day_cols_raw].sum()

items_with_leading_zeros = []
for item_id, row in item_level_raw.iterrows():
    vals = row.to_numpy()
    if vals[0] == 0:
        first_nonzero = vals.nonzero()[0]
        if len(first_nonzero) > 0 and first_nonzero[0] > 30:
            items_with_leading_zeros.append(item_id)

print(f"{len(items_with_leading_zeros)} of {len(item_level_raw)} items had a real leading-zero stretch (>30 days)")

test_item = 'FOODS_1_004'
before = item_level_raw.loc[test_item]
after = long_df[long_df['item_id'] == test_item].sort_values('date')

print(f"\nVerifying on {test_item}:")
print("First 10 raw values:      ", before.values[:10])
print("Trimmed start date:       ", after['date'].min())
print("First 10 trimmed sales:   ", after['sales'].head(10).tolist())

714 of 1437 items had a real leading-zero stretch (>30 days)

Verifying on FOODS_1_004:
First 10 raw values:       [0 0 0 0 0 0 0 0 0 0]
Trimmed start date:        2012-03-07 00:00:00
First 10 trimmed sales:    [1, 48, 50, 53, 56, 36, 49, 81, 54, 74]


**Confirmed:** 714 of 1,437 items (~49.7%) had a genuine pre-launch zero stretch. `FOODS_1_004` went from 10 leading zeros to a clean start at 2012-03-07, with real demand immediately — the fix works as intended.

Note: some already-active items also show extended *mid-series* zero gaps (e.g. stockouts) — these are a different phenomenon from pre-launch padding and are intentionally **not** trimmed here, since they're a legitimate feature of the demand series that Phase 2's structural-break detection is designed to catch.

### Launch-Date Distribution

Visualizing when each item's (trimmed) series actually starts confirms items enter the dataset at different points over time — consistent with staggered product launches rather than a data error.

In [ ]:
launch_dates = long_df.groupby('item_id')['date'].min()
launch_dates.hist(bins=30)

<Axes: >

<Figure size 640x480 with 1 Axes>

## 5. ABC-XYZ Segmentation

Run **after** the leading-zero fix, so variance and volume estimates reflect real demand, not zero-padded pre-launch periods.

- **ABC** — classifies items by cumulative contribution to total sales volume (A = top ~70%, B = next ~20%, C = remaining ~10%).
- **XYZ** — classifies items by coefficient of variation of demand (X = low variability, Y = moderate, Z = high/intermittent).

A/X and A/Y items (high-volume, low-to-moderate variability) are carried forward as the candidate set for structural-break detection — breaks are most reliably detectable and commercially relevant in this segment. C/Z items are excluded here as too noisy for reliable break detection.

In [ ]:
item_stats = long_df.groupby('item_id')['sales'].agg(
    total_sales='sum',
    mean_sales='mean',
    std_sales='std'
).reset_index()
item_stats['cv'] = item_stats['std_sales'] / item_stats['mean_sales']  # coefficient of variation

# ABC: by volume contribution
item_stats = item_stats.sort_values('total_sales', ascending=False).reset_index(drop=True)
item_stats['cum_pct'] = item_stats['total_sales'].cumsum() / item_stats['total_sales'].sum()
item_stats['abc_class'] = pd.cut(item_stats['cum_pct'], bins=[0, 0.7, 0.9, 1.0], labels=['A', 'B', 'C'])

# XYZ: by variability
item_stats['xyz_class'] = pd.cut(item_stats['cv'], bins=[0, 0.5, 1.0, np.inf], labels=['X', 'Y', 'Z'])

item_stats['segment'] = item_stats['abc_class'].astype(str) + item_stats['xyz_class'].astype(str)
print(item_stats['segment'].value_counts())

segment
CZ    364
BY    363
CY    281
AY    280
BZ     66
AX     57
AZ     23
BX      3
Name: count, dtype: int64


In [ ]:
candidates = item_stats[item_stats['segment'].isin(['AX', 'AY'])]['item_id'].tolist()
print(f"{len(candidates)} items selected for structural break analysis (AX + AY)")

337 items selected for structural break analysis (AX + AY)


## 6. Classical Time-Series Diagnostics

Stationarity (Augmented Dickey-Fuller test) and seasonal decomposition on a representative item (`FOODS_1_004`, a confirmed leading-zero item, post-trim) to establish the descriptive baseline before any break detection.

In [ ]:
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
import matplotlib.pyplot as plt

test_item = 'FOODS_1_004'
series = long_df[long_df['item_id'] == test_item].set_index('date')['sales']

result = adfuller(series)
print(f"ADF Statistic: {result[0]:.4f}, p-value: {result[1]:.4f}")
# p < 0.05 -> stationary

decomposition = seasonal_decompose(series, model='additive', period=7)  # weekly seasonality
decomposition.plot()
plt.savefig("../outputs/figures/foods_1_004_decomposition.png")
plt.show()

ADF Statistic: -4.4880, p-value: 0.0002


<Figure size 640x480 with 4 Axes>

No flat pre-launch stretch remains in the decomposition — the series reflects real demand behavior from day one, confirming the trim fix end-to-end.

## 7. Stationarity Across the Full Candidate Set

ADF test applied to all 337 AX/AY candidates (not just the one sample item) — this is the number actually needed for Phase 2, and for the project's reported baseline.

In [ ]:
adf_results = []

for item_id in candidates:
    series = long_df[long_df['item_id'] == item_id].set_index('date')['sales']
    if len(series) < 30:  # skip items with too little history to test meaningfully
        continue
    try:
        result = adfuller(series)
        adf_results.append({'item_id': item_id, 'adf_stat': result[0], 'p_value': result[1]})
    except Exception as e:
        print(f"Failed on {item_id}: {e}")

adf_df = pd.DataFrame(adf_results)
adf_df['is_stationary'] = adf_df['p_value'] < 0.05

print(f"Candidates list length: {len(candidates)}")
print(f"Items actually tested:  {len(adf_df)}")
print(adf_df['is_stationary'].value_counts())
print(f"{adf_df['is_stationary'].mean()*100:.1f}% of items are stationary at 5% significance")

Candidates list length: 337
Items actually tested:  337
is_stationary
True     287
False     50
Name: count, dtype: int64
85.2% of items are stationary at 5% significance


## 8. Save Outputs

Persist the segmented candidate set, segmentation table (with stationarity merged in), and ADF results — these are the inputs Phase 2 (structural break detection) reads from.

In [ ]:
item_stats = item_stats.merge(adf_df[['item_id', 'p_value', 'is_stationary']], on='item_id', how='left')

item_stats.to_csv("../data/interim/item_segmentation.csv", index=False)
adf_df.to_csv("../data/interim/adf_results.csv", index=False)
long_df[long_df['item_id'].isin(candidates)].to_csv("../data/interim/scoped_sales_candidates.csv", index=False)

print("Saved: item_segmentation.csv, adf_results.csv, scoped_sales_candidates.csv")

## Conclusion

Scoped M5's California/FOODS category (1,437 items) down to **337 high-volume, low-to-moderate-variability items** (ABC-XYZ segmentation) as the subset for structural-break analysis. Along the way, found and fixed a real data-quality issue: **714 of 1,437 items (~49.7%) had a leading-zero stretch caused by the product not yet having launched**, not genuine zero demand — left uncorrected, this would have inflated variance estimates and caused false break-detection at launch dates in Phase 2.

After the fix, **85.2% of the 337-item candidate subset is stationary at the 5% significance level (287 stationary / 50 non-stationary)**, giving a clean, verified baseline for structural-break detection in Phase 2.

**Next:** `01_structural_break_detection.ipynb` — CUSUM-based structural break detection across the 337-item candidate set, validated against real calendar events.